In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [10]:
# poly_id in img_path is not the same as the poly_id my workflow expects
# my workflow’s expected poly_id should actually contain lookup.site_poly_uuid.
# in the final output, the poly_id column should be overwritten with values from lookup.site_poly_uuid.
lookup_path = "../data/debug/poly_id_lookup_202603171728_c2.xlsx"
img_path = "../data/debug/c2_10_projects_example.csv"
out_path = "../data/debug/c2_10_projects_example_mapped.csv"

lookup = pd.read_excel(lookup_path)
img = pd.read_csv(img_path)

# Keep only needed columns and drop blank rows
lookup = lookup[["site_poly_uuid", "poly_id"]].dropna().drop_duplicates()

# Basic uniqueness checks
dup_poly = lookup["poly_id"].duplicated().sum()
dup_uuid = lookup["site_poly_uuid"].duplicated().sum()

print(f"Duplicate poly_id rows in lookup: {dup_poly}")
print(f"Duplicate site_poly_uuid rows in lookup: {dup_uuid}")

vals = img["poly_id"].astype(str).str.strip()
lookup_poly = set(lookup["poly_id"].astype(str).str.strip())
lookup_uuid = set(lookup["site_poly_uuid"].astype(str).str.strip())

poly_match_count = vals.isin(lookup_poly).sum()
uuid_match_count = vals.isin(lookup_uuid).sum()

print(f"Matches to lookup.poly_id: {poly_match_count} / {len(vals)}")
print(f"Matches to lookup.site_poly_uuid: {uuid_match_count} / {len(vals)}")

# Case A: img.poly_id really is poly_id
if poly_match_count >= uuid_match_count:
    print("Using img.poly_id as true poly_id and mapping site_poly_uuid onto it.")
    out = img.merge(
        lookup.drop_duplicates(subset=["poly_id"]),
        on="poly_id",
        how="left"
    )

# Case B: img.poly_id is actually site_poly_uuid mislabeled
else:
    print("Using img.poly_id as mislabeled site_poly_uuid and mapping true poly_id from lookup.")
    out = img.merge(
        lookup.drop_duplicates(subset=["site_poly_uuid"]),
        left_on="poly_id",
        right_on="site_poly_uuid",
        how="left"
    )

    out = out.drop(columns=["poly_id_x"]).rename(columns={"poly_id_y": "poly_id"})

# Optional: reorder columns
cols = out.columns.tolist()
if "site_poly_uuid" in cols and "poly_id" in cols:
    front = ["poly_id", "site_poly_uuid"]
    rest = [c for c in cols if c not in front]
    out = out[front + rest]

# Report unmapped rows
if "site_poly_uuid" in out.columns:
    unmapped = out["site_poly_uuid"].isna().sum()
else:
    unmapped = out["poly_id"].isna().sum()

print(f"Unmapped rows: {unmapped}")

out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Duplicate poly_id rows in lookup: 0
Duplicate site_poly_uuid rows in lookup: 0
Matches to lookup.poly_id: 3196 / 3200
Matches to lookup.site_poly_uuid: 0 / 3200
Using img.poly_id as true poly_id and mapping site_poly_uuid onto it.
Unmapped rows: 4
Saved: ../data/debug/c2_10_projects_example_mapped.csv


In [12]:

img_df = pd.read_csv("../data/imagery_availability/c2_10_projects_example_mapped.csv", dtype={"datetime": "string"})

print(img_df["datetime"].head(10).tolist())
print(pd.to_datetime(img_df["datetime"], errors="coerce", utc=True).head(10))

['5/26/23', '5/29/23', '5/29/23', '3/11/24', '4/18/24', '6/3/24', '6/22/24', '6/22/24', '9/26/24', '10/17/24']
0   2023-05-26 00:00:00+00:00
1   2023-05-29 00:00:00+00:00
2   2023-05-29 00:00:00+00:00
3   2024-03-11 00:00:00+00:00
4   2024-04-18 00:00:00+00:00
5   2024-06-03 00:00:00+00:00
6   2024-06-22 00:00:00+00:00
7   2024-06-22 00:00:00+00:00
8   2024-09-26 00:00:00+00:00
9   2024-10-17 00:00:00+00:00
Name: datetime, dtype: datetime64[ns, UTC]


/var/folders/jk/ytyjl7sj4n981fvqgd4_9v5w0000gp/T/ipykernel_61684/3815629622.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  print(pd.to_datetime(img_df["datetime"], errors="coerce", utc=True).head(10))
